In [7]:
%pip uninstall torch torchvision torchaudio

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
COCO_CLASSES = {
    0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane',
    5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light',
    10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench',
    14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow',
    20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack',
    25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee',
    30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat',
    35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket',
    39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon',
    45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange',
    50: 'brocolli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut',
    55: 'cake', 56: 'chair', 57: 'couch', 58: 'potted plant', 59: 'bed',
    60: 'dining table', 61: 'toilet', 62: 'tv', 63: 'laptop', 64: 'mouse',
    65: 'remote', 66: 'keyboard', 67: 'cell phone', 68: 'microwave', 69: 'oven',
    70: 'toaster', 71: 'sink', 72: 'refrigerator', 73: 'book', 74: 'clock',
    75: 'vase', 76: 'scissors', 77: 'teddy bear', 78: 'hair drier', 79: 'toothbrush'
}


In [ ]:
import torch
import cv2
import pandas as pd
import os
from ultralytics import YOLO
from pathlib import Path
CLASSES = {
    0: 'person',
    1: 'bicycle',
    2: 'car',
    16: 'dog',
    25: 'umbrella',
}

def main():
    device = 'cuda:0'
    # 2. Загрузка модели (YOLOv8n - самая быстрая и точная для CPU/GPU)
    model = YOLO('yolov8n.pt')
    model.to(device)
    # 3. Настройка классов из COCO (индексы соответствуют yolo_classes.txt)
    images_dir = Path('lab13')
    image_files = sorted([f for f in images_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    results_table = []
    for img_path in image_files:
        # Инференс
        results = model(img_path, device=device, verbose=False, conf=0.4, iou=0.45)
        result = results[0]
        # Подсчет объектов по классам
        counts = {name: 0 for name in CLASSES.values()}
        if result.boxes is not None and len(result.boxes) > 0:
            boxes = result.boxes
            for i in range(len(boxes)):
                cls_id = int(boxes.cls[i].item())
                if cls_id in CLASSES:
                    counts[CLASSES[cls_id]] += 1
        row = {'Файл': img_path.name}
        for cls_name in CLASSES.values():
            row[f'{cls_name}_найдено'] = counts[cls_name]
        results_table.append(row)

        # Визуализация и сохранение
        img_cv = cv2.imread(str(img_path))
        if result.boxes is not None:
            for box in result.boxes:
                cls_id = int(box.cls[0].item())
                conf = float(box.conf[0].item())
                if cls_id in CLASSES and conf >= 0.4:
                    x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
                    cls_name = CLASSES[cls_id]
                    cv2.rectangle(img_cv, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    cv2.putText(img_cv, f'{cls_name} {conf:.2f}', (x1, y1 - 10),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        out_path = f"result_13/result_{img_path.name}"
        cv2.imwrite(out_path, img_cv)
    df = pd.DataFrame(results_table)
    df.to_excel('result_13/lab13_report.xlsx', index=False)
    df.to_csv('result_13/lab13_report.csv', index=False)

if __name__ == '__main__':
    main()